# Test I: Multi-Class Classification — Classical CNN

**Task:** Classify strong gravitational lensing images into 3 classes:
- `no` — no substructure
- `sphere` — subhalo substructure
- `vort` — vortex substructure

**Approach:** ResNet-18 adapted for single-channel 150×150 input, trained with SGD + cosine annealing.

**Evaluation:** ROC curve + AUC score (one-vs-rest)

In [ ]:
import os, sys
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
import warnings; warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Dataset

In [ ]:
CLASS_NAMES = ['no', 'sphere', 'vort']
DATA_DIR = 'data'

class LensingDataset(Dataset):
    def __init__(self, root_dir, augment=False):
        self.samples, self.labels = [], []
        self.augment = augment
        for ci, cn in enumerate(CLASS_NAMES):
            d = os.path.join(root_dir, cn)
            for f in sorted(os.listdir(d)):
                if f.endswith('.npy'):
                    self.samples.append(os.path.join(d, f))
                    self.labels.append(ci)
        self.labels = np.array(self.labels)
        print(f'Loaded {len(self.samples)} samples from {root_dir}')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img = np.load(self.samples[idx]).astype(np.float32)
        label = self.labels[idx]
        if self.augment:
            if np.random.rand() > 0.5: img = img[:, :, ::-1].copy()
            if np.random.rand() > 0.5: img = img[:, ::-1, :].copy()
            k = np.random.randint(0, 4)
            img = np.rot90(img, k, axes=(1, 2)).copy()
        return torch.from_numpy(img), label

train_dataset = LensingDataset(os.path.join(DATA_DIR, 'train'), augment=True)
val_dataset = LensingDataset(os.path.join(DATA_DIR, 'val'), augment=False)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=0, pin_memory=True)

In [ ]:
# Visualize samples
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, cn in enumerate(CLASS_NAMES):
    idx = np.where(train_dataset.labels == i)[0][0]
    img, _ = train_dataset[idx]
    axes[i].imshow(img[0], cmap='hot')
    axes[i].set_title(cn); axes[i].axis('off')
plt.tight_layout(); plt.show()

## 2. Model — ResNet-18

In [ ]:
from model_cnn import build_resnet18

model = build_resnet18(num_classes=3).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 3. Training

In [ ]:
NUM_EPOCHS = 50
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

best_val_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    model.train()
    rl, c, t = 0., 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        rl += loss.item() * imgs.size(0)
        c += (out.argmax(1) == labels).sum().item()
        t += imgs.size(0)

    model.eval()
    vrl, vc, vt = 0., 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            loss = criterion(out, labels)
            vrl += loss.item() * imgs.size(0)
            vc += (out.argmax(1) == labels).sum().item()
            vt += imgs.size(0)

    scheduler.step()
    history['train_loss'].append(rl/t); history['val_loss'].append(vrl/vt)
    history['train_acc'].append(c/t); history['val_acc'].append(vc/vt)

    if vc/vt > best_val_acc:
        best_val_acc = vc/vt
        torch.save(model.state_dict(), 'weights/best_cnn.pt')

    print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS} | '
          f'Train Loss: {rl/t:.4f} Acc: {c/t:.4f} | '
          f'Val Loss: {vrl/vt:.4f} Acc: {vc/vt:.4f}')

print(f'\nBest Val Accuracy: {best_val_acc:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()
ax1.set_title('Loss'); ax1.grid(alpha=0.3)
ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend()
ax2.set_title('Accuracy'); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Evaluation — ROC Curve & AUC

In [ ]:
model.load_state_dict(torch.load('weights/best_cnn.pt'))
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        probs = torch.softmax(model(imgs), dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)
all_labels_bin = label_binarize(all_labels, classes=[0, 1, 2])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#e41a1c', '#377eb8', '#4daf4a']
for i, cn in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(all_labels_bin[:, i], all_probs[:, i])
    ax.plot(fpr, tpr, color=colors[i], lw=2, label=f'{cn} (AUC={auc(fpr,tpr):.4f})')

macro_auc = roc_auc_score(all_labels_bin, all_probs, multi_class='ovr', average='macro')
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curves — Classical CNN (Macro AUC = {macro_auc:.4f})')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/roc_cnn.png', dpi=150)
plt.show()

print(f'\nPer-class AUC:')
for i, cn in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(all_labels_bin[:, i], all_probs[:, i])
    print(f'  {cn}: {auc(fpr,tpr):.4f}')
print(f'Macro AUC: {macro_auc:.4f}')